In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Load dataset
df = pd.read_csv('dataset/car-dataset-v1.csv')

# Menampilkan 5 baris pertama dan informasi dataset
print("Ukuran dataset awal:", df.shape)
df.info()
df.head()

Ukuran dataset awal: (4340, 8)
<class 'pandas.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   name           4340 non-null   str  
 1   year           4340 non-null   int64
 2   selling_price  4340 non-null   int64
 3   km_driven      4340 non-null   int64
 4   fuel           4340 non-null   str  
 5   seller_type    4340 non-null   str  
 6   transmission   4340 non-null   str  
 7   owner          4340 non-null   str  
dtypes: int64(3), str(5)
memory usage: 271.4 KB


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [2]:
# Memeriksa baris duplikat
duplicates = df.duplicated().sum()
print(f"Jumlah baris duplikat eksak: {duplicates}")

# Menghapus baris duplikat (mencegah Data Leakage)
df = df.drop_duplicates()
print("Ukuran dataset setelah hapus duplikat:", df.shape)

Jumlah baris duplikat eksak: 763
Ukuran dataset setelah hapus duplikat: (3577, 8)


In [3]:
# 1. Ekstraksi Merek (Brand) dari kolom 'name'
df['brand'] = df['name'].apply(lambda x: x.split()[0])

# 2. Hitung Umur Mobil (car_age) dari kolom 'year'
current_year = 2020  # Tahun rilis maksimum di dataset
df['car_age'] = current_year - df['year']

# Hapus kolom asli 'name' dan 'year' karena sudah diwakilkan
df = df.drop(columns=['name', 'year'])
df.head()

,selling_price,km_driven,fuel,seller_type,transmission,owner,brand,car_age
0,60000,70000,Petrol,Individual,Manual,First Owner,Maruti,13
1,135000,50000,Petrol,Individual,Manual,First Owner,Maruti,13
2,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai,8
3,250000,46000,Petrol,Individual,Manual,First Owner,Datsun,3
4,450000,141000,Diesel,Individual,Manual,Second Owner,Honda,6


In [4]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Ordinal Encoding khusus untuk kolom 'owner' (karena punya hirarki urutan)
owner_order = [
    [
        'Test Drive Car',
        'First Owner',
        'Second Owner',
        'Third Owner',
        'Fourth & Above Owner',
    ]
]
ordinal_enc = OrdinalEncoder(categories=owner_order)
df['owner_encoded'] = ordinal_enc.fit_transform(df[['owner']])
df = df.drop(columns=['owner'])

# 2. One-Hot Encoding untuk fitur nominal (brand, fuel, seller_type, transmission)
df = pd.get_dummies(
    df,
    columns=['brand', 'fuel', 'seller_type', 'transmission'],
    drop_first=True,
)

df.head()

,selling_price,km_driven,car_age,owner_encoded,brand_Audi,brand_BMW,brand_Chevrolet,brand_Daewoo,brand_Datsun,brand_Fiat,...,brand_Toyota,brand_Volkswagen,brand_Volvo,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual
0,60000,70000,13,1.0,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,True
1,135000,50000,13,1.0,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,True
2,600000,100000,8,1.0,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,True
3,250000,46000,3,1.0,False,False,False,False,True,False,...,False,False,False,False,False,False,True,True,False,True
4,450000,141000,6,2.0,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,True


In [5]:
from sklearn.model_selection import train_test_split

# Memisahkan Fitur (X) dan Target (y)
X = df.drop(columns=['selling_price'])
y = df['selling_price']

# Data Splitting (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

print("Jumlah Data Latih (Train):", len(X_train))
print("Jumlah Data Uji (Test):", len(X_test))

Jumlah Data Latih (Train): 2861
Jumlah Data Uji (Test): 716


In [6]:
from sklearn.preprocessing import StandardScaler

# 1. Transformasi Logaritmik pada Target (selling_price)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. Transformasi Logaritmik pada Fitur 'km_driven'
X_train = X_train.copy()
X_test = X_test.copy()

X_train['km_driven'] = np.log1p(X_train['km_driven'])
X_test['km_driven'] = np.log1p(X_test['km_driven'])

# 3. Standardisasi Fitur Numerik (Fit HANYA di Train Set)
scaler = StandardScaler()
num_cols = ['car_age', 'km_driven', 'owner_encoded']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [7]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Pelatihan Model Linear Regression
LR = LinearRegression().fit(X_train, y_train_log)

# 2. Pelatihan Model Gradient Boosting Regressor
GBR = GradientBoostingRegressor(random_state=184).fit(X_train, y_train_log)

# Prediksi (Hasil masih dalam skala logaritma)
pred_LR_log = LR.predict(X_test)
pred_GBR_log = GBR.predict(X_test)

# 3. Mengembalikan Hasil Prediksi ke Skala Nominal Rupiah Asli (np.expm1)
pred_LR = np.expm1(pred_LR_log)
pred_GBR = np.expm1(pred_GBR_log)

# Evaluasi Metrik berdasarkan nilai Rupiah Asli
df_results = pd.DataFrame(
    {
        'MAE (Rupiah)': [
            mean_absolute_error(y_test, pred_LR),
            mean_absolute_error(y_test, pred_GBR),
        ],
        'MSE': [
            mean_squared_error(y_test, pred_LR),
            mean_squared_error(y_test, pred_GBR),
        ],
        'R2 Score': [r2_score(y_test, pred_LR), r2_score(y_test, pred_GBR)],
    },
    index=['Linear Regression (Improved)', 'GradientBoostingRegressor (Improved)'],
)

df_results

,MAE (Rupiah),MSE,R2 Score
Linear Regression (Improved),137137.923278,5.666063e+10,0.825726
GradientBoostingRegressor (Improved),138500.153947,7.138053e+10,0.780451


In [8]:
import joblib

# Menyimpan model Linear Regression (karena R2 Score paling tinggi ~0.825)
joblib.dump(LR, 'car_price_model_improved.joblib')
print("Model berhasil disimpan!")

Model berhasil disimpan!
